# Content-Based Recommender System

In this notebook, we build a content-based recommender using the MovieLens dataset.

Unlike collaborative filtering, content-based recommendation suggests items based on their own characteristics. In this case, we use movie genres as item features.

The main idea is:
- represent each movie using its genres
- build a user profile from the movies the user liked
- recommend unseen movies that are most similar to that user profile

Contributors: Andrés Ramírez, Yago Moreno

## 0. Import the libraries (Setup)

In [1]:
#Import the necessary libraries

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load the Data

In [2]:
try:
    ratings = pd.read_csv("ml-latest-small/ratings.csv")
    movies = pd.read_csv("ml-latest-small/movies.csv")
except FileNotFoundError:
    ratings = pd.read_csv("ratings.csv")
    movies = pd.read_csv("movies.csv")

## 2. Temporal Train-Test Split

We use a temporal split so that recommendations are generated only from past interactions.

In [3]:
ratings = ratings.sort_values("timestamp").copy()

cutoff = int(len(ratings) * 0.8)
train = ratings.iloc[:cutoff].copy()
test = ratings.iloc[cutoff:].copy()

print("Train size:", len(train))
print("Test size:", len(test))

Train size: 80668
Test size: 20168


## 3. Feature Engineering

We represent each movie using its genres. The genre column is transformed into a binary feature matrix.

In [4]:
movies_features = movies.copy()
movies_features["genres"] = movies_features["genres"].str.replace("|", " ", regex=False)

genre_matrix = movies_features["genres"].str.get_dummies(sep=" ")
genre_matrix.index = movies_features["movieId"]

genre_matrix.head()

,(no,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,...,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,genres,listed)
movieId,,,,,,,,,,,,,,,,,,,,,
1,0,0,1,1,1,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
5,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 4. Item-Item Similarity

We compute cosine similarity between movies based on their genre vectors.

In [5]:
item_similarity = cosine_similarity(genre_matrix)

item_similarity = pd.DataFrame(
    item_similarity,
    index=genre_matrix.index,
    columns=genre_matrix.index
)

item_similarity.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.774597,0.316228,0.258199,0.447214,0.0,0.316228,0.632456,0.0,0.258199,...,0.447214,0.316228,0.316228,0.447214,0.0,0.670820,0.774597,0.00000,0.316228,0.447214
2,0.774597,1.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.816497,0.0,0.333333,...,0.000000,0.000000,0.000000,0.000000,0.0,0.288675,0.333333,0.00000,0.000000,0.000000
3,0.316228,0.000000,1.000000,0.816497,0.707107,0.0,1.000000,0.000000,0.0,0.000000,...,0.353553,0.000000,0.500000,0.000000,0.0,0.353553,0.408248,0.00000,0.000000,0.707107
4,0.258199,0.000000,0.816497,1.000000,0.577350,0.0,0.816497,0.000000,0.0,0.000000,...,0.288675,0.408248,0.816497,0.000000,0.0,0.288675,0.333333,0.57735,0.000000,0.577350
5,0.447214,0.000000,0.707107,0.577350,1.000000,0.0,0.707107,0.000000,0.0,0.000000,...,0.500000,0.000000,0.707107,0.000000,0.0,0.500000,0.577350,0.00000,0.000000,1.000000


## 5. Similar Movies Function

For a given movie, we retrieve the most similar movies according to the genre-based similarity matrix.

In [6]:
def get_similar_movies(movie_id, item_similarity, movies, top_n=10):
    if movie_id not in item_similarity.index:
        return pd.DataFrame(columns=["movieId", "title", "similarity"])

    sim_scores = item_similarity.loc[movie_id].sort_values(ascending=False)

    # Remove the movie itself
    sim_scores = sim_scores.iloc[1:top_n+1]

    similar_movies = pd.DataFrame({
        "movieId": sim_scores.index,
        "similarity": sim_scores.values
    })

    similar_movies = similar_movies.merge(movies[["movieId", "title"]], on="movieId")

    return similar_movies[["movieId", "title", "similarity"]]

## 6. Example: Similar Movies

In [7]:
example_movie = movies["movieId"].iloc[0]

print("Selected movie:", movies[movies["movieId"] == example_movie]["title"].values[0])
get_similar_movies(example_movie, item_similarity, movies, top_n=10)

Selected movie: Toy Story (1995)


,movieId,title,similarity
0,103755,Turbo (2013),1.0
1,4886,"Monsters, Inc. (2001)",1.0
2,91355,Asterix and the Vikings (Astérix et les Viking...,1.0
3,136016,The Good Dinosaur (2015),1.0
4,53121,Shrek the Third (2007),1.0
5,65577,"Tale of Despereaux, The (2008)",1.0
6,2294,Antz (1998),1.0
7,3754,"Adventures of Rocky and Bullwinkle, The (2000)",1.0
8,4016,"Emperor's New Groove, The (2000)",1.0
9,45074,"Wild, The (2006)",1.0


## 7. Recommend Movies for a User

We recommend movies that are similar to the ones the user previously liked.

In [8]:
def recommend_for_user(user_id, train_ratings, item_similarity, movies, top_n=10, min_rating=4.0):
    user_ratings = train_ratings[train_ratings["userId"] == user_id].copy()
    liked_movies = user_ratings[user_ratings["rating"] >= min_rating]["movieId"]

    if liked_movies.empty:
        return pd.DataFrame(columns=["movieId", "title", "score"])

    scores = pd.Series(dtype=float)

    for movie_id in liked_movies:
        if movie_id in item_similarity.index:
            similar_scores = item_similarity.loc[movie_id]
            scores = scores.add(similar_scores, fill_value=0)

    scores = scores.sort_values(ascending=False)

    seen_movies = user_ratings["movieId"].unique()
    scores = scores[~scores.index.isin(seen_movies)]

    recommendations = pd.DataFrame({
        "movieId": scores.index,
        "score": scores.values
    })

    recommendations = recommendations.merge(movies[["movieId", "title"]], on="movieId")

    return recommendations.head(top_n)[["movieId", "title", "score"]]

## 8. Example: User Recommendations

In [9]:
example_user = train["userId"].iloc[0]

print(f"Recommendations for user {example_user}")
recommend_for_user(example_user, train, item_similarity, movies, top_n=10, min_rating=4.0)

Recommendations for user 429


,movieId,title,score
0,4956,"Stunt Man, The (1980)",21.734106
1,72142,Love Exposure (Ai No Mukidashi) (2008),21.591775
2,496,What Happened Was... (1994),21.483813
3,5666,"Rules of Attraction, The (2002)",21.483813
4,6788,Angie (1994),21.185801
5,96975,LOL (2012),21.185801
6,96945,Love Lasts Three Years (L'amour dure trois ans...,21.185801
7,6765,Under the Tuscan Sun (2003),21.185801
8,6750,Anything Else (2003),21.185801
9,6711,Lost in Translation (2003),21.185801


## 9. Prediction Function for Notebook 5

To compare this recommender with the others in Notebook 5, we generate two outputs for each user-item pair in the test set:

- `score_content`: a ranking score
- `predicted_rating_content`: a predicted rating for RMSE and MAE

In [10]:
def predict_content_metrics(user_id, movie_id, train_data, item_similarity, min_rating=4.0):
    user_ratings = train_data[train_data["userId"] == user_id].copy()
    liked_movies = user_ratings[user_ratings["rating"] >= min_rating][["movieId", "rating"]]

    if liked_movies.empty:
        return np.nan, np.nan

    if movie_id not in item_similarity.index:
        return np.nan, np.nan

    similarities = []
    ratings_list = []

    for _, row in liked_movies.iterrows():
        liked_movie = row["movieId"]
        liked_rating = row["rating"]

        if liked_movie in item_similarity.columns:
            sim = item_similarity.loc[movie_id, liked_movie]
            similarities.append(sim)
            ratings_list.append(liked_rating)

    if len(similarities) == 0:
        return np.nan, np.nan

    similarities = np.array(similarities)
    ratings_list = np.array(ratings_list)

    # Ranking score
    score_content = similarities.mean()

    # Predicted rating
    if similarities.sum() > 0:
        predicted_rating_content = np.dot(similarities, ratings_list) / similarities.sum()
    else:
        predicted_rating_content = ratings_list.mean()

    return score_content, predicted_rating_content

## 10. Compute Predictions on the Full Test Set

In [11]:
pred_content = test[["userId", "movieId", "rating"]].copy()

pred_content[["score_content", "predicted_rating_content"]] = pred_content.apply(
    lambda row: pd.Series(
        predict_content_metrics(
            row["userId"],
            row["movieId"],
            train,
            item_similarity,
            min_rating=4.0
        )
    ),
    axis=1
)

global_mean = train["rating"].mean()

pred_content["score_content"] = pred_content["score_content"].fillna(0)
pred_content["predicted_rating_content"] = pred_content["predicted_rating_content"].fillna(global_mean)

pred_content.head()

,userId,movieId,rating,score_content,predicted_rating_content
79691,495,72998,5.0,0.260112,4.783460
79564,495,2762,4.5,0.340262,4.660192
79601,495,4993,0.5,0.108660,4.825495
79709,495,89492,5.0,0.487505,4.680301
79551,495,2028,4.5,0.434230,4.641142


## 12. Interpretation

This recommender suggests movies that are similar in genre to the movies the user already liked.

Its main advantage is interpretability and independence from other users.

Its main limitation is overspecialization, since it tends to recommend items very similar to past preferences.

## 13. Conclusion

In this notebook, we implemented a content-based recommender using movie genres as item features.

First, movies were represented through a binary genre matrix. Then, cosine similarity was used to measure similarity between items. Based on this similarity structure, the model was able to recommend movies similar to those previously liked by each user.

To make the model suitable for the final project evaluation, we also generated two outputs for each user-item pair in the test set:
- a ranking score (`score_content`)
- a predicted rating (`predicted_rating_content`)

This allows the model to be compared with the other recommenders in Notebook 5 using both error-based metrics and ranking metrics.

Overall, the content-based approach is simple, interpretable, and useful when item features are available. However, it may suffer from overspecialization and depends strongly on the quality of the item representation.

## 14. Export Predictions for Notebook 5

In [12]:
pred_content.to_csv("predictions_content.csv", index=False)

print("File exported: predictions_content.csv")
pred_content.head()

File exported: predictions_content.csv


,userId,movieId,rating,score_content,predicted_rating_content
79691,495,72998,5.0,0.260112,4.783460
79564,495,2762,4.5,0.340262,4.660192
79601,495,4993,0.5,0.108660,4.825495
79709,495,89492,5.0,0.487505,4.680301
79551,495,2028,4.5,0.434230,4.641142
